In [0]:
%sql
DESCRIBE TABLE observatorio_dev.bronze.plantas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
bronze_table = "observatorio_dev.bronze.plantas"
silver_table = "observatorio_dev.silver.plantas"

In [0]:
bronze_df = spark.table(bronze_table)

print("Registros en Bronze plantas:", bronze_df.count())

In [0]:
silver_df = (
    bronze_df
    .select(
        F.col("id"),

        F.to_date(F.col("fecha")).alias("fecha"),

        F.upper(F.trim(F.col("codigo_duracion"))).alias("codigo_duracion"),
        F.upper(F.trim(F.col("codigo_planta"))).alias("codigo_planta"),
        F.upper(F.trim(F.col("nombre_planta"))).alias("nombre_planta"),
        F.upper(F.trim(F.col("codigo_sic_agente"))).alias("codigo_sic_agente"),

        F.regexp_replace(F.col("cap_efectiva_neta"), ",", ".").cast("double").alias("cap_efectiva_neta"),

        F.to_date(F.col("fpo")).alias("fpo"),

        F.upper(F.trim(F.col("codigo_sub_area_operativa"))).alias("codigo_sub_area_operativa"),
        F.upper(F.trim(F.col("codigo_area_operativa"))).alias("codigo_area_operativa"),
        F.upper(F.trim(F.col("tipo_despacho_recurso"))).alias("tipo_despacho_recurso"),
        F.upper(F.trim(F.col("tipo_clasificacion"))).alias("tipo_clasificacion"),
        F.upper(F.trim(F.col("tipo_generacion"))).alias("tipo_generacion"),

        F.col("source_file_name"),
        F.col("source_file_path"),
        F.col("ingestion_timestamp"),
        F.col("load_date"),

        F.current_timestamp().alias("silver_created_at"),
        F.current_timestamp().alias("silver_updated_at")
    )
    .filter(F.col("codigo_planta").isNotNull())
)

In [0]:
raw_null_keys_count = (
    bronze_df
    .filter(F.col("codigo_planta").isNull())
    .count()
)

print("Registros Bronze sin codigo_planta:", raw_null_keys_count)

In [0]:
duplicates_df = (
    silver_df
    .groupBy("codigo_planta", "fecha")
    .count()
    .filter(F.col("count") > 1)
)

print("Plantas duplicadas antes de deduplicar:", duplicates_df.count())

In [0]:
window_spec = (
    Window
    .partitionBy("codigo_planta", "fecha")
    .orderBy(
        F.col("fecha").desc_nulls_last(),
        F.col("load_date").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last(),
        F.col("id").desc_nulls_last()
    )
)

silver_df = (
    silver_df
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Registros después de deduplicar:", silver_df.count())

In [0]:
new_records_count = silver_df.count()

print("Registros a procesar:", new_records_count)

if new_records_count == 0:
    print("No hay registros para cargar en Silver.")

else:
    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.codigo_planta = source.codigo_planta
            AND target.fecha = source.fecha
            """
        )
        .whenMatchedUpdate(set={
            "fecha": "source.fecha",
            "codigo_duracion": "source.codigo_duracion",
            "nombre_planta": "source.nombre_planta",
            "codigo_sic_agente": "source.codigo_sic_agente",
            "cap_efectiva_neta": "source.cap_efectiva_neta",
            "fpo": "source.fpo",
            "codigo_sub_area_operativa": "source.codigo_sub_area_operativa",
            "codigo_area_operativa": "source.codigo_area_operativa",
            "tipo_despacho_recurso": "source.tipo_despacho_recurso",
            "tipo_clasificacion": "source.tipo_clasificacion",
            "tipo_generacion": "source.tipo_generacion",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_updated_at": "source.silver_updated_at"
        })
        .whenNotMatchedInsert(values={
            "id": "source.id",
            "fecha": "source.fecha",
            "codigo_duracion": "source.codigo_duracion",
            "codigo_planta": "source.codigo_planta",
            "nombre_planta": "source.nombre_planta",
            "codigo_sic_agente": "source.codigo_sic_agente",
            "cap_efectiva_neta": "source.cap_efectiva_neta",
            "fpo": "source.fpo",
            "codigo_sub_area_operativa": "source.codigo_sub_area_operativa",
            "codigo_area_operativa": "source.codigo_area_operativa",
            "tipo_despacho_recurso": "source.tipo_despacho_recurso",
            "tipo_clasificacion": "source.tipo_clasificacion",
            "tipo_generacion": "source.tipo_generacion",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_created_at": "source.silver_created_at",
            "silver_updated_at": "source.silver_updated_at"
        })
        .execute()
    )

    print("MERGE de plantas ejecutado correctamente.")

In [0]:
%sql
SELECT 
  MIN(s.id) AS min_id, 
  COUNT(*) AS total_records,
  COUNT(*) FILTER (WHERE s.id = b.id) AS records_same_id_in_bronze
FROM observatorio_dev.silver.plantas s
LEFT JOIN observatorio_dev.bronze.plantas b
  ON s.id = b.id